# VideoAgent

> Notebook generated from [`examples/multimodal/03_video_agent.py`](../../examples/multimodal/03_video_agent.py).

| Field | Value |
|------|-------|
| **Dataset** | ActivityNet (embedded, 3 clips) |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** Per clip: transcript, frame descriptions with timestamps, fused summary.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
VideoAgent — Video pipeline (frames + transcript + fusion)
==============================================================
Component: SPEC-MM-AGT-003 / prismal.agents.multimodal.video_agent

Dataset: ActivityNet-style captions (temporal video descriptions)
  • ActivityNet Captions contains 20k videos with descriptions over
    time (start, end, sentence) — one of the most-used benchmarks
    for video understanding.
  • Reference: http://activity-net.org/challenges/2017/captioning.html
  • Why: the `VideoAgent` produces a `VideoResult(transcript,
    frame_descriptions, summary, …)` that closely matches the structure
    of ActivityNet annotations. We use synthetic clips whose captions
    are pre-defined.

Component description:
  1. `MediaValidator` validates that the file is a video (MP4/WebM).
  2. `frame_extractor_fn(video_path, fps, max_frames)` runs FFmpeg
     inside `SandboxExecutor` in production — here we mock it.
  3. `VisionAgent.analyze(frame_path)` describes each extracted
     frame (parallelized with `asyncio.gather`).
  4. `transcribe_fn(video_path)` extracts the audio from the video (FFmpeg →
     `AudioAgent.process`) in production — here we inject it.
  5. `fusion_fn(transcript, frame_descriptions)` joins everything into the
     final summary (multimodal LLM in production).

Usage:
    uv run python examples/multimodal/03_video_agent.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import struct
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import AsyncMock

from prismal.agents.multimodal import (
    FrameDescription,
    VideoAgent,
    VideoResult,
    VisionAgent,
    VisionResult,
)

## Dataset: 3 synthetic ActivityNet-style clips

In [ ]:
CLIPS = [
    {
        "id": "anet_001",
        "filename": "cooking_pasta.mp4",
        "duration_s": 24.0,
        "fps": 1.0,
        "frame_captions": [
            "A pot of water boiling on a stove",
            "Hands pouring dry spaghetti into the boiling pot",
            "Someone stirring the pasta with a wooden spoon",
            "Plating the cooked pasta with red sauce",
        ],
        "transcript": "Today we're making a quick spaghetti dinner from scratch.",
    },
    {
        "id": "anet_002",
        "filename": "skateboard_trick.mp4",
        "duration_s": 12.0,
        "fps": 2.0,
        "frame_captions": [
            "A skateboarder approaching a stair set",
            "Mid-air ollie above the stairs",
            "Landing the trick on the asphalt below",
        ],
        "transcript": "Watch this kickflip down the seven stairs!",
    },
    {
        "id": "anet_003",
        "filename": "guitar_lesson.mp4",
        "duration_s": 30.0,
        "fps": 0.5,
        "frame_captions": [
            "Close-up of guitarist's hand forming a G chord",
            "Strumming pattern demonstrated slowly",
            "Hand moving to a C chord",
        ],
        "transcript": "In this lesson we'll cover the G to C transition.",
    },
]

## Fake MP4 file (sufficient for `MediaValidator.sniff()`)

In [ ]:
def _make_fake_mp4() -> bytes:
    """Minimal MP4: the detector looks for 'ftyp' at offset 4."""
    # 4 size bytes + 'ftypisom' + padding
    box = b"ftypisom\x00\x00\x02\x00isomiso2avc1mp41"
    return struct.pack(">I", len(box) + 4) + box + b"\x00" * 256

## Fake collaborators — no FFmpeg, no VLM, no Whisper

In [ ]:
def make_frame_extractor(out_dir: Path, frames_per_clip: dict[str, int]):
    """Mock frame extractor: writes N empty PNGs and returns their paths."""
    import zlib

    def _png(label_byte: int) -> bytes:
        sig = b"\x89PNG\r\n\x1a\n"

        def _chunk(tag: bytes, data: bytes) -> bytes:
            crc = zlib.crc32(tag + data)
            return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", crc)

        ihdr = struct.pack(">IIBBBBB", 8, 8, 8, 2, 0, 0, 0)
        row = b"\x00" + bytes([label_byte, 0, 0]) * 8
        idat = zlib.compress(row * 8, 9)
        return sig + _chunk(b"IHDR", ihdr) + _chunk(b"IDAT", idat) + _chunk(b"IEND", b"")

    async def _extract(video: Path, fps: float, max_frames: int) -> list[Path]:  # noqa: ARG001
        clip_id = video.stem
        n_frames = min(frames_per_clip.get(clip_id, 3), max_frames)
        out: list[Path] = []
        for i in range(n_frames):
            p = out_dir / f"{clip_id}_frame_{i:03d}.png"
            p.write_bytes(_png(i * 30 % 255))
            out.append(p)
        return out

    return _extract


def make_vision_agent(captions_by_clip: dict[str, list[str]]) -> VisionAgent:
    """VisionAgent wraps a vision_fn that looks at the file name."""

    async def _vision(image, prompt: str) -> str:  # noqa: ARG001
        if not isinstance(image, Path):
            return "(mock vision)"
        clip_id = image.stem.rsplit("_frame_", 1)[0]
        idx = int(image.stem.rsplit("_", 1)[-1])
        captions = captions_by_clip.get(clip_id, [])
        return captions[idx] if idx < len(captions) else f"frame {idx}"

    return VisionAgent(vision_fn=_vision)


def make_transcribe_fn(transcripts_by_clip: dict[str, str]):
    async def _transcribe(video: Path) -> str:
        return transcripts_by_clip.get(video.stem, "")

    return _transcribe


def make_fusion_fn():
    """Deterministic fusion: concat audio + frames with labels."""

    async def _fuse(transcript: str, frames: list[FrameDescription]) -> str:
        lines = [f"AUDIO: {transcript}"] if transcript else []
        for fr in frames:
            lines.append(f"FRAME {fr.timestamp_s:.1f}s: {fr.description}")
        return "\n".join(lines)

    return _fuse


async def main() -> None:
    print("=" * 70)
    print("VideoAgent · frames + audio + fusion pipeline over ActivityNet-style clips")
    print("=" * 70)

    with TemporaryDirectory(prefix="prismal_vid_") as tmp:
        tmp_dir = Path(tmp)

        # Create the mock MP4s once.
        video_paths: dict[str, Path] = {}
        frames_per_clip = {}
        captions_by_clip = {}
        transcripts_by_clip = {}
        for clip in CLIPS:
            clip_id = Path(clip["filename"]).stem
            p = tmp_dir / clip["filename"]
            p.write_bytes(_make_fake_mp4())
            video_paths[clip["id"]] = p
            frames_per_clip[clip_id] = len(clip["frame_captions"])
            captions_by_clip[clip_id] = clip["frame_captions"]
            transcripts_by_clip[clip_id] = clip["transcript"]

        agent = VideoAgent(
            vision_agent=make_vision_agent(captions_by_clip),
            frame_extractor_fn=make_frame_extractor(tmp_dir, frames_per_clip),
            transcribe_fn=make_transcribe_fn(transcripts_by_clip),
            fusion_fn=make_fusion_fn(),
            degrade_gracefully=True,
        )

        for clip in CLIPS:
            print("\n" + "─" * 70)
            print(f"Clip: {clip['id']}  ({clip['filename']})  fps={clip['fps']}")
            print("─" * 70)
            result: VideoResult = await agent.summarize(
                video_paths[clip["id"]],
                fps=clip["fps"],
                max_frames=10,
            )
            print(f"  transcript        : {result.transcript}")
            print(f"  frames processed  : {result.total_frames_processed}")
            print("  frame descriptions:")
            for fr in result.frame_descriptions:
                print(f"    t={fr.timestamp_s:.1f}s · {fr.description}")
            print("  fused summary:")
            for line in result.summary.splitlines():
                print(f"    {line}")

        # Validation rejects a non-video blob.
        print("\n" + "─" * 70)
        print("Validation: non-video file produces empty VideoResult (fallback)")
        print("─" * 70)
        bogus = tmp_dir / "not_a_video.bin"
        bogus.write_bytes(b"garbage")
        result = await agent.summarize(bogus, fps=1.0, max_frames=3)
        assert result.summary == "" and result.total_frames_processed == 0
        print("  ← the agent returned an empty VideoResult without extracting frames")

    print("\n" + "=" * 70)
    print("OK — VideoAgent works with mocked FFmpeg/VLM (no real sandbox)")
    print("=" * 70)


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()